In [2]:
import pandas as pd
from itertools import permutations, combinations
import uuid
import numpy as np

In [3]:
SELECTED_STOP=pd.read_excel("../Data/BUS/SELECTED_STOP.xlsx",engine="openpyxl")
SELECTED_STOP.head()

,SELECTED_ROUTE_ID,DISTRICT,STOP_ID,STOP_NAMEC,STOP_TYPE
0,1,中西区,122,"金钟 - 太古广场, 金钟道",商业
1,2,中西区,127,"林士街, 德辅道中",工业
2,3,中西区,354,"正街, 德辅道西",居民
3,4,中西区,393,"联邦新楼, 卑路乍街",居民
4,5,湾仔区,3045,"箕琏坊, 蓝塘道",居民


In [4]:
# 存储结果
transit_data = []

# 按区域分组
grouped = SELECTED_STOP.groupby("DISTRICT")

transit_id = 1  # 递增的 TRANSIT_ID
route_id = 1  # 递增的 SELECTED_ROUTE_ID
route_map = {}  # 用于存储站点对的路线 ID


# 创建一个空的 DataFrame 用于存储结果
stop_pairs = []

# 按照 DISTRICT 分组，并在每个分组内选取所有可能的两两组合
for district, group in SELECTED_STOP.groupby("DISTRICT"):
    for (on_stop, off_stop) in combinations(group.itertuples(index=False), 2):
        stop_pairs.append({
            "DISTRICT": district,
            "ON_STOP_ID": on_stop.STOP_ID,
            "ON_STOP_NAMES": on_stop.STOP_NAMEC,
            "ON_STOP_TYPE": on_stop.STOP_TYPE,
            "OFF_STOP_ID": off_stop.STOP_ID,
            "OFF_STOP_NAMES": off_stop.STOP_NAMEC,
            "OFF_STOP_TYPE": off_stop.STOP_TYPE,
        })

# 转换为 DataFrame
transit_df = pd.DataFrame(stop_pairs)


In [5]:
transit_df

,DISTRICT,ON_STOP_ID,ON_STOP_NAMES,ON_STOP_TYPE,OFF_STOP_ID,OFF_STOP_NAMES,OFF_STOP_TYPE
0,中西区,122,"金钟 - 太古广场, 金钟道",商业,127,"林士街, 德辅道中",工业
1,中西区,122,"金钟 - 太古广场, 金钟道",商业,354,"正街, 德辅道西",居民
2,中西区,122,"金钟 - 太古广场, 金钟道",商业,393,"联邦新楼, 卑路乍街",居民
3,中西区,127,"林士街, 德辅道中",工业,354,"正街, 德辅道西",居民
4,中西区,127,"林士街, 德辅道中",工业,393,"联邦新楼, 卑路乍街",居民
5,中西区,354,"正街, 德辅道西",居民,393,"联邦新楼, 卑路乍街",居民
6,沙田区,12409,乌溪沙站,居民,851,沙角邨,居民
7,沙田区,12409,乌溪沙站,居民,858,车公庙,居民
8,沙田区,12409,乌溪沙站,居民,10000002,火炭骏洋邨,工业
9,沙田区,851,沙角邨,居民,858,车公庙,居民


In [6]:
STOP_XY=pd.read_excel("../Data/BUS/STOP_XY.xlsx",engine="openpyxl")
STOP_XY.columns

Index(['STOP_ID', 'STOP_TYPE', 'X', 'Y', 'LAST_UPDATE_DATE', 'STOP_CODE'], dtype='object')

In [7]:
# 关联 STOP_XY 以获取坐标
transit_df = transit_df.merge(STOP_XY[['STOP_ID', 'X', 'Y']], left_on='ON_STOP_ID', right_on='STOP_ID', how='left')
transit_df.rename(columns={'X': 'ON_STOP_X', 'Y': 'ON_STOP_Y'}, inplace=True)
transit_df.drop(columns=['STOP_ID'], inplace=True)

transit_df = transit_df.merge(STOP_XY[['STOP_ID', 'X', 'Y']], left_on='OFF_STOP_ID', right_on='STOP_ID', how='left')
transit_df.rename(columns={'X': 'OFF_STOP_X', 'Y': 'OFF_STOP_Y'}, inplace=True)
transit_df.drop(columns=['STOP_ID'], inplace=True)

In [8]:
transit_df['DISTANCE'] = np.sqrt((transit_df['ON_STOP_X'] - transit_df['OFF_STOP_X'])**2 + 
                                  (transit_df['ON_STOP_Y'] - transit_df['OFF_STOP_Y'])**2)

In [9]:
transit_df.head()

,DISTRICT,ON_STOP_ID,ON_STOP_NAMES,ON_STOP_TYPE,OFF_STOP_ID,OFF_STOP_NAMES,OFF_STOP_TYPE,ON_STOP_X,ON_STOP_Y,OFF_STOP_X,OFF_STOP_Y,DISTANCE
0,中西区,122,"金钟 - 太古广场, 金钟道",商业,127,"林士街, 德辅道中",工业,835057,815462,833903,816342,1451.246361
1,中西区,122,"金钟 - 太古广场, 金钟道",商业,354,"正街, 德辅道西",居民,835057,815462,832646,816570,2653.410070
2,中西区,122,"金钟 - 太古广场, 金钟道",商业,393,"联邦新楼, 卑路乍街",居民,835057,815462,831204,815956,3884.539226
3,中西区,127,"林士街, 德辅道中",工业,354,"正街, 德辅道西",居民,833903,816342,832646,816570,1277.510470
4,中西区,127,"林士街, 德辅道中",工业,393,"联邦新楼, 卑路乍街",居民,833903,816342,831204,815956,2726.462360


In [11]:
Converted_Fare=pd.read_excel("../Data/BUS/Converted_Fare.xlsx",engine="openpyxl")
Converted_Fare.columns

Index(['ROUTE_ID', 'ROUTE_SEQ', 'ON_SEQ', 'OFF_SEQ', 'PRICE',
       'LAST_UPDATE_DATE', 'ROUTE_NAMES', 'COMPANY_CODE', 'STOP_SEQ_x',
       'STOP_NAMES_x', 'ON_SEQ_STOP_ID', 'STOP_SEQ_y', 'STOP_NAMES_y',
       'OFF_SEQ_STOP_ID', 'STOP_ID_x', 'ON_SEQ_STOP_X', 'ON_SEQ_STOP_Y',
       'STOP_ID_y', 'OFF_SEQ_STOP_X', 'OFF_SEQ_STOP_Y'],
      dtype='object')

In [12]:
fare_df = Converted_Fare[['ON_SEQ_STOP_ID', 'OFF_SEQ_STOP_ID', 'PRICE']]
fare_df = fare_df.groupby(['ON_SEQ_STOP_ID', 'OFF_SEQ_STOP_ID'])['PRICE'].min().reset_index()
transit_df = transit_df.merge(fare_df, left_on=['ON_STOP_ID', 'OFF_STOP_ID'], right_on=['ON_SEQ_STOP_ID', 'OFF_SEQ_STOP_ID'], how='left')
transit_df.drop(columns=['ON_SEQ_STOP_ID', 'OFF_SEQ_STOP_ID'], inplace=True)

In [13]:
transit_df.head()

,DISTRICT,ON_STOP_ID,ON_STOP_NAMES,ON_STOP_TYPE,OFF_STOP_ID,OFF_STOP_NAMES,OFF_STOP_TYPE,ON_STOP_X,ON_STOP_Y,OFF_STOP_X,OFF_STOP_Y,DISTANCE,PRICE
0,中西区,122,"金钟 - 太古广场, 金钟道",商业,127,"林士街, 德辅道中",工业,835057,815462,833903,816342,1451.246361,NaN
1,中西区,122,"金钟 - 太古广场, 金钟道",商业,354,"正街, 德辅道西",居民,835057,815462,832646,816570,2653.410070,NaN
2,中西区,122,"金钟 - 太古广场, 金钟道",商业,393,"联邦新楼, 卑路乍街",居民,835057,815462,831204,815956,3884.539226,4.8
3,中西区,127,"林士街, 德辅道中",工业,354,"正街, 德辅道西",居民,833903,816342,832646,816570,1277.510470,NaN
4,中西区,127,"林士街, 德辅道中",工业,393,"联邦新楼, 卑路乍街",居民,833903,816342,831204,815956,2726.462360,NaN


In [14]:
transit_df.to_excel("../Data/BUS/GENERATED_ROUTE.xlsx",index=False,engine='openpyxl')

### Compute Cost Index

In [18]:
GENERATED_ROUTE=pd.read_excel("../Data/BUS/GENERATED_ROUTE_LB.xlsx",engine="openpyxl")
GENERATED_ROUTE.columns

Index(['DISTRICT', 'ON_STOP_ID', 'ON_STOP_NAMES', 'ON_STOP_TYPE',
       'OFF_STOP_ID', 'OFF_STOP_NAMES', 'OFF_STOP_TYPE', 'ON_STOP_X',
       'ON_STOP_Y', 'OFF_STOP_X', 'OFF_STOP_Y', 'DISTANCE', 'PRICE',
       'ROUTE_NAMES', 'COMPANY_CODE'],
      dtype='object')

In [ ]:
# 计算每个区域的总距离
district_total_distance = GENERATED_ROUTE.groupby("DISTRICT")["DISTANCE"].transform("sum")

# 计算成本指数
GENERATED_ROUTE["COST_INDEX"] = (GENERATED_ROUTE["PRICE"] * GENERATED_ROUTE["DISTANCE"]) / district_total_distance

GENERATED_ROUTE.to_excel("../Data/BUS/GENERATED_ROUTE_LB.xlsx", index=False, engine="openpyxl")
